# Create raster layers

## About this notebook

The source of truth for all raster layer definitions is `raster_config.yaml`. Most layers run straight from config with no manual steps required and are not documented here.

This notebook documents only layers that needed **custom preprocessing** before running the standard processor — merging tiles, resampling, clipping to boundaries, assigning a missing CRS, etc. These steps are kept for reproducibility and reference.

To process layers that require no preprocessing, use the **working cell** at the bottom of this notebook.

## Setup

### Library import

In [ ]:
import sys
from pathlib import Path

import geopandas as gpd
import rasterio as rio
from rasterio.mask import mask

sys.path.append("../src")

from data_processing.process_layers import process_prestyled_raster, process_raster_layers
from data_processing.utils import clip_rasters_by_vector, merge_tifs, resample_raster

project_root = Path("/Users/sofia/Documents/Repos/esa/data-processing")
config_path = str(project_root / "src" / "raster_config.yaml")

## Preprocessing records

#### Somalia - Water

In [ ]:
# Preprocessing for Somalia - WorldCover dataset
folder_to_merge = Path(
    "../data/raw/Water/Somalia/Rasters/"
    "4_GDAWater_ESAWorldCover_GrasslandCropland_10m_v200_N00E039_Somalia_2021_10042025/"
)
merged_file = Path("../data/raw/Water/Somalia/Rasters/WorldCover.tif")
resampled_file = Path("../data/raw/Water/Somalia/Rasters/WorldCover_resampled.tif")
clipped_folder = Path("../data/raw/Water/Somalia/Rasters/Clipped/")
gadm_file = Path("../data/raw/gadm_410-adm_0/gadm_410-adm_0.shp")
country_name = "Somalia"

# Step 1: Merge TIFs
merge_tifs(folder_to_merge, merged_file)

# Step 2: Resample merged raster (memory-efficient)
resample_raster(merged_file, resampled_file, scale_factor=5)

# Step 3: Create Somalia vector from GADM shapefile
gdf = gpd.read_file(gadm_file)
somalia_gdf = gdf[gdf["COUNTRY"] == country_name]
temp_vector = Path("../data/temp_somalia.shp")
somalia_gdf.to_file(temp_vector)

# Step 4: Clip the resampled raster to Somalia
temp_raster_folder = Path("../data/temp_resample/")
temp_raster_folder.mkdir(exist_ok=True)
(resampled_file).rename(temp_raster_folder / resampled_file.name)

clip_rasters_by_vector(temp_raster_folder, temp_vector, clipped_folder)

In [ ]:
# Clip the MeanAnnualPrecipitation raster to Somalia
input_file = Path(
    "../data/raw/Water/Somalia/Rasters/"
    "3_GDAWater_GroundwaterResourcesEstimation_MeanAnnualGroundwaterStorageBalance_Multi-annual_Somalia_2003-2024_10042025.tif"
)
gadm_file = Path("../data/raw/gadm_410-adm_0/gadm_410-adm_0.shp")
clipped_folder = Path("../data/raw/Water/Somalia/Rasters/Clipped/")
country_name = "Somalia"

# Create output folder
clipped_folder.mkdir(parents=True, exist_ok=True)

# Extract Somalia vector
gdf = gpd.read_file(gadm_file)
somalia_gdf = gdf[gdf["COUNTRY"] == country_name]

# Direct clipping without temp files
output_file = clipped_folder / input_file.name

with rio.open(input_file) as src:
    # Clip with Somalia boundary
    out_image, out_transform = mask(src, somalia_gdf.geometry, crop=True, nodata=src.nodata)

    # Update metadata
    out_meta = src.meta.copy()
    out_meta.update({
        "height": out_image.shape[1],
        "width": out_image.shape[2],
        "transform": out_transform
    })

    # Save clipped raster
    with rio.open(output_file, "w", **out_meta) as dest:
        dest.write(out_image)

print(f"Clipped raster saved to: {output_file}")

In [ ]:
process_raster_layers(
    config_path=config_path,
    layer_keys=[
        "somalia_worldcover",
        "somalia_groundwater_storage",
    ],
    dry_run=False,
    isolate_process=True,
    upload_layer_keys=[],
)

#### West Africa - Water

In [ ]:
# Assign CRS to raster (no reprojection needed)
input_file = (
    "../data/raw/Water/West Africa/Rasters/"
    "1_GDAWater_Precipitation_MeanAnnualPrecipitation_Multi-annual_WestAfrica_2003-2024_23052025.tif"
)
output_file = (
    "../data/raw/Water/West Africa/Rasters/"
    "1_GDAWater_Precipitation_MeanAnnualPrecipitation_Multi-annual_WestAfrica_2003-2024_23052025_prj.tif"
)

# Create output directory if it doesn't exist
Path(output_file).parent.mkdir(parents=True, exist_ok=True)

with rio.open(input_file) as src:
    print(f"Original CRS: {src.crs}")
    print(f"Original dimensions: {src.width} x {src.height}")

    # Check if bounds look like geographic coordinates (lat/lon)
    print(f"Bounds: {src.bounds}")

    # Copy all data and metadata, just assign the CRS
    kwargs = src.meta.copy()
    kwargs.update({
        'crs': 'EPSG:4326',  # Assign the CRS without reprojecting
        'compress': 'LZW',
        'tiled': True,
        'blockxsize': 256,
        'blockysize': 256
    })

    # Copy the raster data as-is, just with proper CRS
    with rio.open(output_file, 'w', **kwargs) as dst:
        dst.write(src.read())  # Copy all bands directly
    print(f"CRS assigned and saved to: {output_file}")

In [ ]:
# West Africa groundwater availability clipping
input_folder = Path("../data/raw/Water/West Africa/Rasters/To_clip/")
vector_file = Path("../data/raw/Water/West Africa/boundaries/west_africa_countries.shp")
clipped_folder = Path("../data/raw/Water/West Africa/Rasters/Clipped/")

clip_rasters_by_vector(input_folder, vector_file, clipped_folder)

In [ ]:
process_raster_layers(
    config_path=config_path,
    layer_keys=[
        "west_africa_precipitation",
        "west_africa_groundwater_availability",
    ],
    upload_layer_keys=[],
)

## Process new layers

Use the cell below to process layers that run straight from config (no preprocessing needed).

**Workflow:**
1. Add the layer definition to `raster_config.yaml`
2. Edit `layer_keys` below and run
3. If preprocessing was needed, move those cells to the **Preprocessing records** section above with a markdown header explaining what was done
4. Revert this cell before committing — do not accumulate processed layer keys here

In [ ]:
# WORKING CELL — edit layer_keys, run, then revert. Do not commit with real values.
process_raster_layers(
    config_path=config_path,
    layer_keys=["your_layer_key_here"],
    upload_layer_keys=[],
)